# 01 — Data Exploration (Real AAPL Data)

## What we're doing here
Now that we understand the concepts from the hand-crafted fixtures, we apply the same pipeline to **real market data**.  
This notebook loads the downloaded AAPL OHLCV files and gives us a first look before running any algorithm.

## The four timeframes

| File | Timeframe | Typical use in the pipeline |
|------|-----------|-----------------------------|
| `1wk.csv` | Weekly | HTF bias — the big picture |
| `1d.csv`  | Daily   | Zone detection (primary) |
| `4h.csv`  | 4-Hour  | Zone detection (intraday) |
| `1h.csv`  | 1-Hour  | Entry trigger / LTF confirmation |

## Key difference from fixtures
The fixtures were hand-crafted — every scenario had a known answer.  
Real data is **messy**: gaps, low-volume periods, earnings spikes, stock splits.  
We will spot those here and handle them before passing data to the detection pipeline.

## 1. Setup

In [6]:
%pip install pandas numpy plotly nbformat pydantic PyYAML --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Load all four timeframes

In [7]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from utils.data_loader import load_all_timeframes, TIMEFRAMES 
from utils.data_downloader import download_all

download_all()  # ← set force=True to re-download all data

SYMBOL = "USDJPY=X"   # ← change to any downloaded symbol

data = load_all_timeframes(SYMBOL, align=True)

print(f"Symbol : {SYMBOL}")
print(f"Aligned start date: {list(data.values())[0].index[0].date()}")
print()
print(f"{'TF':<6} {'Rows':>6}  {'First':>12}  {'Last':>12}  {'NaNs':>5}")
print("-" * 54)
for tf, df in data.items():
    print(f"{tf:<6} {len(df):>6}  "
          f"{str(df.index[0].date()):>12}  "
          f"{str(df.index[-1].date()):>12}  "
          f"{df.isnull().sum().sum():>5}")


No data — MATIC-USD [1d]
$MATIC-USD: possibly delisted; no price data found  (period=2y)
No data — MATIC-USD [4h]
$MATIC-USD: possibly delisted; no price data found  (period=2y)
No data — MATIC-USD [1h]
No data — UNI-USD [1d]
$UNI-USD: possibly delisted; no price data found  (period=2y)
No data — UNI-USD [4h]
$UNI-USD: possibly delisted; no price data found  (period=2y)
No data — UNI-USD [1h]


Symbol : USDJPY=X
Aligned start date: 2024-06-16

TF       Rows         First          Last   NaNs
------------------------------------------------------
1wk       104    2024-06-16    2026-06-07      0
1d        581    2024-06-13    2026-06-12      0
4h       3174    2024-06-13    2026-06-12      0
1h      12259    2024-06-12    2026-06-12      0


## 3. Inspect the daily data

In [8]:
df_1d = data["1d"]
print(f"Shape  : {df_1d.shape}")
print(f"Columns: {list(df_1d.columns)}")
print()
df_1d.head(10)

Shape  : (581, 5)
Columns: ['open', 'high', 'low', 'close', 'volume']



,open,high,low,close,volume
Datetime,,,,,
2024-06-13 00:00:00+00:00,156.796997,157.311005,156.570007,157.121002,0
2024-06-14 00:00:00+00:00,157.121002,158.257004,156.794998,157.322998,0
2024-06-16 00:00:00+00:00,157.516998,157.662003,157.488007,157.552002,0
2024-06-17 00:00:00+00:00,157.552994,157.960007,157.149994,157.694000,0
2024-06-18 00:00:00+00:00,157.694000,158.229004,157.511993,157.856995,0
2024-06-19 00:00:00+00:00,157.862000,158.130997,157.595001,157.957001,0
2024-06-20 00:00:00+00:00,157.955002,158.949997,157.949997,158.936005,0
2024-06-21 00:00:00+00:00,158.936005,159.835007,158.664001,159.774994,0
2024-06-23 00:00:00+00:00,159.710007,159.904999,159.673004,159.895004,0


## 4. Basic statistics — daily close

In [9]:
df_1d[["open","high","low","close","volume"]].describe().round(2)

,open,high,low,close,volume
count,581.00,581.00,581.00,581.00,581.0
mean,151.79,152.36,151.13,151.78,0.0
std,5.62,5.52,5.70,5.63,0.0
min,140.69,140.90,139.57,140.76,0.0
25%,147.15,147.81,146.59,147.15,0.0
50%,151.98,152.74,151.45,151.93,0.0
75%,156.91,157.35,156.29,156.92,0.0
max,161.59,161.95,161.30,161.59,0.0


## 5. Multi-timeframe overview

All four timeframes stacked on a shared time axis — weekly for HTF bias, daily for zone detection, 4h and 1h for entry context.


In [10]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"

fig2 = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    row_heights=[0.30, 0.25, 0.25, 0.20],
    subplot_titles=["Weekly (1wk)", "Daily (1d)", "4-Hour (4h)", "1-Hour (1h)"],
    vertical_spacing=0.03,
)

for row, tf in enumerate(["1wk", "1d", "4h", "1h"], start=1):
    d = data[tf]
    fig2.add_trace(go.Candlestick(
        x=d.index,
        open=d["open"], high=d["high"],
        low=d["low"],   close=d["close"],
        increasing_line_color=COLOR_BULL,
        decreasing_line_color=COLOR_BEAR,
        name=tf, showlegend=True,
    ), row=row, col=1)

fig2.update_layout(
    title=f"{SYMBOL} — Multi-Timeframe Overview (1wk / 1d / 4h / 1h)",
    height=960,
    plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.02),
)
for ax in ["xaxis","xaxis2","xaxis3","xaxis4","yaxis","yaxis2","yaxis3","yaxis4"]:
    fig2.update_layout(**{ax: dict(gridcolor=GRID, showgrid=True,
                                   zeroline=False, linecolor=GRID)})
for r in [1, 2, 3, 4]:
    fig2.update_xaxes(rangeslider_visible=False, row=r, col=1)
fig2.show()


## 6. What's next

| Notebook | Topic |
|----------|-------|
| `02_candle_primitives.ipynb` | Label each real candle: bullish / bearish / doji / base |
| `03_atr.ipynb`               | Compute ATR(14) on the real series |
| `04_base_detection.ipynb`    | Detect base clusters on real AAPL |
| …                              | Continue the full S&D pipeline |

From notebook 02 onward every notebook uses `data["1d"]` (and other timeframes) instead of `fixtures.csv`.